In [21]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import TensorBoardLogger

In [82]:
class DosDataset(Dataset):

    def __init__(self, param_file, data_loc):
        dataset = np.load(f'data/{param_file}.npz')
        params_arr = dataset['params']
        dos_arr = dataset['dos']
        
        self.params_arr = torch.from_numpy(params_arr).float()
        self.dos_arr = torch.from_numpy(dos_arr).float()
    
    def __getitem__(self, index):
        return self.params_arr[index], self.dos_arr[index]
    
    def __len__(self):
        return self.params_arr.size(dim=0)
   
train_dataset = DosDataset("train-set", data_loc)
val_dataset = DosDataset("val-set", data_loc)

print(train_dataset.dos_arr.size())
print(val_dataset.params_arr.size())

torch.Size([106120, 301])
torch.Size([13265, 3])


In [96]:
class LitNeuralNet(pl.LightningModule):
    def __init__(self, layer_sizes, lr = 0.01):
        super(LitNeuralNet, self).__init__()
        
        modules = []
        for i in range(len(layer_sizes) - 1):
            modules.append(nn.Linear(layer_sizes[i], layer_sizes[i + 1]))
            
            if i != len(layer_sizes) - 2:
                modules.append(nn.ReLU())
        
        self.forward_prop = nn.Sequential(*modules)
        self.learning_rate = lr
    
    def training_step(self, batch, batch_idx):
        params, dos = batch
        
        # Forward pass
        predicted = self.forward_prop(params)
        loss = F.mse_loss(predicted, dos)
        
        #log to tensorboard
        self.log("train_loss", loss)
        return loss
    
    def validation_step(self, batch, batch_idx):
        params, dos = batch
        
        # Forward pass
        predicted = self.forward_prop(params)
        loss = F.mse_loss(predicted, dos)
        
        #log to tensorboard
        self.log("val_loss", loss)
        return loss
        
    def configure_optimizers(self):
        return torch.optim.Adam(model.parameters(), lr=self.learning_rate)

In [99]:
#Try a fast dev run
layer_sizes = [3, 200, 200, 301]
batch_size = 1000
learning_rate = 0.005

train_loader = DataLoader(train_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

model = LitNeuralNet(layer_sizes, lr = learning_rate)

trainer = pl.Trainer(fast_dev_run = True)
trainer.fit(model, train_dataloaders = train_loader, val_dataloaders = val_loader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Running in fast_dev_run mode: will run a full train, val, test and prediction loop using 1 batch(es).
`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.
`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.
`Trainer(limit_test_batches=1)` was configured so 1 batch will be used.
`Trainer(limit_predict_batches=1)` was configured so 1 batch will be used.
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..

  | Name         | Type       | Params
--------------------------------------------
0 | forward_prop | Sequential | 101 K 
--------------------------------------------
101 K     Trainable params
0         Non-trainable params
101 K     Total params
0.406     Total estimated model params size (MB)


Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [100]:
#100 epoch run
layer_sizes = [3, 200, 200, 301]
batch_size = 1000
learning_rate = 0.005
num_epochs = 100
logger = TensorBoardLogger('nn_logs', name = f'lr={learning_rate}, batch={batch_size}')

train_loader = DataLoader(train_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

model = LitNeuralNet(layer_sizes, lr = learning_rate)

trainer = pl.Trainer(max_epochs = num_epochs, logger = logger)
trainer.fit(model, train_dataloaders = train_loader, val_dataloaders = val_loader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: nn_logs/lr=0.005, batch=1000

  | Name         | Type       | Params
--------------------------------------------
0 | forward_prop | Sequential | 101 K 
--------------------------------------------
101 K     Trainable params
0         Non-trainable params
101 K     Total params
0.406     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

In [103]:
#Try automatic learning rate finder
layer_sizes = [3, 200, 200, 301]
batch_size = 1000
num_epochs = 100
logger = TensorBoardLogger('nn_logs', name = f'lr=auto, batch={batch_size}')

train_loader = DataLoader(train_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

model = LitNeuralNet(layer_sizes)

trainer = Trainer(auto_lr_find=True, max_epochs=num_epochs)
trainer.tune(model, train_dataloaders = train_loader, val_dataloaders = val_loader)

print(f"Found learning rate of : {model.learning_rate}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

Learning rate set to 0.0010964781961431851
Restoring states from the checkpoint path at /project/wyin/jlee/ml-project/dos-prediction/.lr_find_0a00245b-39df-4104-800d-3a4957ece2f8.ckpt


Found learning rate of : 0.0010964781961431851
